# DA003 - Automated Testing

>  **Note**: this notebook is provided for educational purposes, for members of the [Fabric AI Workflows community](https://skool.com/fabricai). All content contained within is protected by Copyright © law. Do not copy or re-distribute. 

This notebook runs automated regression tests against your Machine Performance Data Agent.

What this notebook does:
1. Loads your test questions from the CSV you uploaded to the Lakehouse.
2. Generates the current expected answers by running each question's T-SQL directly against the Lakehouse SQL Analytics Endpoint.
3. Sends each question to your Data Agent and compares its response to the expected answer.
4. Displays a summary and detailed results.

In [ ]:
# Install the Fabric Data Agent SDK for running automated evaluations.
# Run this cell first — the session will restart after installation.
%pip install -q -U fabric-data-agent-sdk

## Setup

Before continuing, make sure you have:

1. Attached your Lakehouse — In the notebook explorer pane, click + Data items and add your DA001_LH_MachineData Lakehouse. This gives the notebook access to the CSV file and the Lakehouse tables.
2. Uploaded the CSV — Upload DA003_Automated_Testing.csv to the Files area of the Lakehouse.
3. Copied your SQL Analytics Endpoint details — In the Lakehouse, go to Settings > SQL analytics endpoint and copy the connection string value. You'll paste it into the Configuration cell below.

In [ ]:
# ============================================================
# CONFIGURATION - Update these values before running
# ============================================================

# The name of your Machine Performance Data Agent (from DA002)
DATA_AGENT_NAME = "DA002 Example Queries"

# Set to None if your Data Agent is in the same workspace as this notebook
WORKSPACE_NAME = None

# "sandbox" tests the unpublished draft, "production" tests the published version
DATA_AGENT_STAGE = "sandbox"

# Lakehouse table name for storing evaluation results
OUTPUT_TABLE_NAME = "da003_evaluation_output"

# SQL Analytics Endpoint details (from your Lakehouse Settings)
SQL_ENDPOINT = "your-endpoint.datawarehouse.fabric.microsoft.com"  # Paste your Server value here
SQL_DATABASE = "DA001_LH_MachineData"  # Your Lakehouse name

## Step 1: Load Test Questions

Load the test questions from the CSV file you uploaded to the Lakehouse. Each row contains a `Question` and the `ExpectedSQL` that produces the correct answer.

In [ ]:
# Load test questions from the CSV file in the Lakehouse Files area
df = (spark.read.format("csv")
      .option("header", "true")
      .option("quote", '"')
      .option("escape", '"')
      .option("multiLine", "true")
      .load("Files/DA003_Automated_Testing.csv"))

print(f"Loaded {df.count()} test questions")
display(df)

## Step 2: Generate Expected Answers

The WFS dataset updates continuously — the Data Agent always works with the most recent 30 days of data. This means we can't store a static expected answer in advance. Instead, we need to run each question's T-SQL right now to get the current correct answer.

To do this, we connect to the Lakehouse SQL Analytics Endpoint. This is the same SQL engine that the Data Agent queries, so we're running the exact same T-SQL — no translation or conversion needed.

The next cell connects to the SQL Analytics Endpoint using your notebook identity (no passwords needed).

In [ ]:
# Connect to the Lakehouse SQL Analytics Endpoint.
# This uses the JDBC driver that's already available in the Spark runtime
# and authenticates using your notebook identity — no passwords needed.

token = notebookutils.credentials.getToken("https://database.windows.net/.default")

props = spark._jvm.java.util.Properties()
props.setProperty("accessToken", token)
props.setProperty("encrypt", "true")
props.setProperty("trustServerCertificate", "false")

jdbc_url = f"jdbc:sqlserver://{SQL_ENDPOINT};databaseName={SQL_DATABASE}"
conn = spark._jvm.java.sql.DriverManager.getConnection(jdbc_url, props)

print(f"Connected to {SQL_DATABASE}")

Now we loop through each test question, run its T-SQL, and store the result as JSON. This gives us a fresh expected answer for every question based on today's data.

The `run_tsql` helper function below handles executing each query and converting the results into a Python-friendly format.

In [ ]:
import json
from pyspark.sql import Row

def run_tsql(connection, tsql):
    """Execute a T-SQL query and return the results as a list of dicts."""
    stmt = connection.createStatement()
    rs = stmt.executeQuery(tsql)

    # Read column names from the result
    meta = rs.getMetaData()
    columns = [meta.getColumnLabel(i + 1) for i in range(meta.getColumnCount())]

    # Read each row into a dict
    rows = []
    while rs.next():
        row = {}
        for i, col in enumerate(columns):
            val = rs.getObject(i + 1)
            # SQL Server returns decimals as Java BigDecimal — convert to Python float
            if val is not None and hasattr(val, 'doubleValue'):
                val = float(val.doubleValue())
            row[col] = val
        rows.append(row)

    rs.close()
    stmt.close()
    return rows

# Loop through each test question and generate the expected answer
question_rows = df.select("Question", "ExpectedSQL").collect()

out = []
for r in question_rows:
    q = r["Question"]
    tsql = r["ExpectedSQL"].strip().rstrip(";")
    result = run_tsql(conn, tsql)
    expected_answer = json.dumps(result, default=str)
    out.append(Row(Question=q, ExpectedSQL=tsql, ExpectedAnswer=expected_answer))

conn.close()

df_expected = spark.createDataFrame(out)
print(f"Generated expected answers for {df_expected.count()} questions")
display(df_expected)

## Step 3: Run Evaluation

Now we have everything we need: a set of questions and their current expected answers. The cell below uses the Fabric Data Agent SDK to:

1. Send each question to your Data Agent.
2. Capture the agent's response.
3. Compare it to the expected answer.
4. Write the results to a Lakehouse table.

Each run gets a unique `evaluation_id`, so you can compare results across multiple runs.

To re-run after making changes to your Data Agent: re-run this cell and the cells below — you don't need to re-run the earlier cells unless you've changed the CSV.

In [ ]:
import pandas as pd
from fabric.dataagent.evaluation import (
    evaluate_data_agent,
    get_evaluation_summary,
    get_evaluation_details
)

# Prepare the ground-truth dataset in the format the evaluator expects
eval_df = (
    df_expected
      .selectExpr(
          "Question as question",
          "ExpectedAnswer as expected_answer"
      )
      .toPandas()
)

# Run the evaluation
evaluation_id = evaluate_data_agent(
    eval_df,
    data_agent_name=DATA_AGENT_NAME,
    workspace_name=WORKSPACE_NAME,
    table_name=OUTPUT_TABLE_NAME,
    data_agent_stage=DATA_AGENT_STAGE
)

print(f"Evaluation run id: {evaluation_id}")
print()

# Show pass/fail summary
display(get_evaluation_summary(OUTPUT_TABLE_NAME))

## Step 4: Detailed Results

The summary above shows pass/fail counts. The detailed view below shows why each question passed or failed — including the Data Agent's response and how it was scored. Look for rows where `evaluation_judgement` is `False` and compare the agent's SQL to the expected SQL.

In [ ]:
details = get_evaluation_details(
    evaluation_id,
    table_name=OUTPUT_TABLE_NAME,
    get_all_rows=True,
    verbose=True
)
display(details)